# 12.16 · LLM 推理优化 / LLM Inference Optimization

> **课程定位 / Where this fits**
> 第 16 课，**Part 12**(本部分收尾)。把 LLM 真正"跑起来、跑得起"的工程最后一公里。
> Lesson 16, **Part 12** (finale). The engineering "last mile" of actually running LLMs affordably.
>
> 训练好的大模型, **推理(生成)又慢又贵**：①**自回归**——必须一个 token 一个 token 串行生成；②模型巨大——每生成一个 token 都要把上百亿参数过一遍；③显存吃紧。让推理**又快又省**是部署 LLM 的核心工程, 直接决定能不能上线、成本多少。本课讲清并**动手演示**最重要的优化: **KV cache(最关键)、量化、投机解码、连续批处理(vLLM)** 等。
> A trained LLM is **slow and expensive at inference**: ① **autoregressive** — generate token by token serially; ② huge — every token runs all billions of params; ③ memory-tight. Making inference **fast and cheap** is core to deploying LLMs and decides feasibility and cost. We explain and **hands-on demo** the key optimizations: **KV cache (most important), quantization, speculative decoding, continuous batching (vLLM)**.
>
> 💼 **实战/面试视角**："KV cache 原理/省了什么 / 量化(int8/int4) / 投机解码 / vLLM/PagedAttention / prefill vs decode" 是 LLM 部署岗高频。
> 💼 **Practical/interview angle:** "KV cache / quantization (int8/int4) / speculative decoding / vLLM/PagedAttention / prefill vs decode" — frequent for LLM deployment roles.

> 📐 **符号约定 / Notation**
> - KV cache —— 缓存历史 token 的 Key/Value, 避免重算 / cache past Keys/Values
> - 量化(quantization) —— 用低位宽(int8/int4)表示权重 / low-bit weights

> 💡 **面试相关 / Interview-relevant**
> - "KV cache 是什么/为什么能加速"（出镜率 ★★★★★）
> - "量化怎么省显存/有什么代价"（★★★★）
> - "投机解码(speculative decoding)原理"（★★★★）
> - "prefill 和 decode 两阶段的区别"（★★★★）
> - "vLLM/PagedAttention/连续批处理"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解 LLM 推理为何慢(自回归+巨大+访存)。
   Understand why LLM inference is slow.
2. **动手演示 KV cache** 如何避免重复计算、加速生成。
   Demo how KV cache avoids recomputation and speeds generation.
3. **动手演示量化** 如何省显存、代价是什么。
   Demo how quantization saves memory and its cost.
4. 了解投机解码、连续批处理等系统级优化。
   Know speculative decoding, continuous batching, etc.

## 目录 / TOC
1. [为什么推理慢 ⭐](#1)
2. [KV Cache：避免重复计算（动手）⭐](#2)
3. [量化：用更少比特（动手）⭐](#3)
4. [投机解码、批处理与小结 ⭐](#4)


<a id="1"></a>
## 1. 为什么推理慢 ⭐ / Why Inference Is Slow

LLM 生成分**两个阶段**(面试要点)：
LLM generation has **two phases** (interview point):
- **Prefill(预填充)**：处理输入 prompt——所有输入 token **一次性并行**过模型(快, 算力密集)。
  **Prefill:** process the input prompt — all input tokens go through **in parallel** (fast, compute-bound).
- **Decode(解码)**：逐个生成新 token——**每次只能生成一个**, 而且生成第 $t$ 个 token 要把整个模型(几十上百 GB 权重)**从显存读一遍**。所以 decode 阶段是**访存密集(memory-bound)**、串行、慢。
  **Decode:** generate new tokens one at a time — **one per step**, and each new token reads the **entire model** (tens-hundreds of GB) from memory. So decode is **memory-bound**, serial, slow.

**核心矛盾**：生成 1000 个 token 就要串行跑 1000 次, 每次都搬一遍庞大的权重。优化推理就是围绕"**少算、少搬、并行更多**"展开。最重要的一招是 **KV cache**。
**Core tension:** generating 1000 tokens means 1000 serial passes, each moving the huge weights. Optimization centers on "**compute less, move less, parallelize more**." The single most important trick is the **KV cache**.


<a id="2"></a>
## 2. KV Cache：避免重复计算（动手）⭐ / KV Cache: Avoid Recomputation

自回归生成时, 每生成一个新 token 都要做自注意力——新 token(作为 query)要和**前面所有 token 的 Key/Value** 交互。
In autoregressive generation, each new token does self-attention — the new token (as query) interacts with the **Keys/Values of all previous tokens**.

**朴素做法(无 cache)**：每生成一步, 都把**整个序列**重新过一遍, 重新算出所有位置的 K/V。但前面那些 token 的 K/V **每一步都一模一样**——重复计算了无数次! 生成 $n$ 个 token 总计算量是 $O(n^2)$。
**Naive (no cache):** at each step, re-run the **entire sequence**, recomputing K/V for all positions. But earlier tokens' K/V are **identical every step** — recomputed countless times! Total work to generate $n$ tokens is $O(n^2)$.

**KV cache**：把已经算过的 K/V **缓存起来**, 每步只算**新 token 自己**的 K/V, 追加到缓存里。总计算量降到 $O(n)$。这是 LLM 推理**最重要、最基本**的优化, 几乎所有推理框架都默认开启。代价：缓存占显存(随序列长度线性增长——这也是后面 PagedAttention 要管的)。
**KV cache:** **cache** the already-computed K/V; each step computes K/V only for the **new token** and appends. Total work drops to $O(n)$. The **most important, most basic** LLM inference optimization, on by default everywhere. Cost: the cache uses memory (grows linearly with sequence length — what PagedAttention later manages).

下面**动手对比**：从零写自注意力生成, 测"无 cache(每步重算全序列)" vs "有 cache" 的计算量与耗时。
Let's **hands-on compare**: from-scratch self-attention generation, measuring compute and time for "no cache (recompute all)" vs "with cache."


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, math, time
import torch, torch.nn as nn, torch.nn.functional as F
sns.set_theme(style="whitegrid"); torch.manual_seed(0)

D, H = 256, 8; dk = D // H
Wq, Wk, Wv = (nn.Linear(D, D) for _ in range(3))
@torch.no_grad()
def attn_step(x_new, K_cache, V_cache):
    """处理一个新token, 返回它的注意力输出, 并把它的K/V加进缓存 / process one new token with KV cache."""
    q = Wq(x_new).view(1, H, dk)
    k = Wk(x_new).view(1, H, dk); v = Wv(x_new).view(1, H, dk)
    K = torch.cat([K_cache, k], 0) if K_cache is not None else k          # 追加新K / append new K
    V = torch.cat([V_cache, v], 0) if V_cache is not None else v
    sc = torch.einsum("ihd,jhd->hij", q, K) / math.sqrt(dk)               # 新query对所有历史K / q vs all K
    out = torch.einsum("hij,jhd->ihd", F.softmax(sc, -1), V).reshape(1, D)
    return out, K, V

@torch.no_grad()
def attn_full(seq):
    """无cache: 对整个序列重算自注意力(模拟每步都这么干) / no cache: full self-attention over the whole sequence."""
    T = seq.shape[0]
    q = Wq(seq).view(T,H,dk); k = Wk(seq).view(T,H,dk); v = Wv(seq).view(T,H,dk)
    sc = torch.einsum("ihd,jhd->hij", q, k) / math.sqrt(dk)
    mask = torch.tril(torch.ones(T,T)).bool()
    sc = sc.masked_fill(~mask, float("-inf"))
    return torch.einsum("hij,jhd->ihd", F.softmax(sc,-1), v).reshape(T,D)

lengths = [50, 100, 200, 400]; t_nocache, t_cache = [], []
for n in lengths:
    seq = torch.randn(n, D)
    # 无cache: 模拟生成n步, 每步都把'到目前为止的序列'整段重算 / no cache: re-run full prefix each step
    t0 = time.time()
    for t in range(1, n+1): _ = attn_full(seq[:t])
    t_nocache.append(time.time()-t0)
    # 有cache: 每步只算新token的K/V, 追加到缓存 / with cache: only new token's K/V each step
    t0 = time.time(); K=V=None
    for t in range(n): _, K, V = attn_step(seq[t:t+1], K, V)
    t_cache.append(time.time()-t0)
fig, ax = plt.subplots(figsize=(7.5,4.5))
ax.plot(lengths, t_nocache, "s-", color="#e67", label="无 KV cache (每步重算全序列, O(n²))")
ax.plot(lengths, t_cache, "o-", color="#39c", label="有 KV cache (每步只算新token, O(n))")
ax.set_xlabel("生成的 token 数 n"); ax.set_ylabel("总耗时 (秒)"); ax.legend()
ax.set_title("KV Cache: 缓存历史K/V → 从 O(n²) 降到 O(n), 序列越长加速越明显")
plt.tight_layout(); plt.show()
for n, a, b in zip(lengths, t_nocache, t_cache):
    print(f"  生成{n}个token: 无cache {a*1000:.0f}ms, 有cache {b*1000:.0f}ms, 加速 {a/b:.1f}x")
print("KV cache: 历史token的K/V每步都一样→缓存复用, 只算新token; LLM推理最基本最重要的优化")


<a id="3"></a>
## 3. 量化：用更少比特（动手）⭐ / Quantization: Fewer Bits

模型推理慢/贵的另一大原因是**模型太大**(显存放不下、访存慢)。**量化(quantization)**：用**更低位宽**表示权重——从 FP32(32位) 或 FP16(16位) 降到 **INT8(8位)** 甚至 **INT4(4位)**。
Another big reason inference is slow/costly is **model size** (won't fit in memory, slow to move). **Quantization:** represent weights in **fewer bits** — from FP32 (32-bit) or FP16 (16-bit) down to **INT8** or even **INT4**.

好处：**显存占用直接减半/减到 1/4/1/8**, 访存更快(decode 是访存密集的, 所以也更快)。代价：**精度损失**(用更少的数表示连续值有舍入误差), 但好的量化方法能把损失降到几乎无感。
Benefits: **memory drops to 1/2, 1/4, 1/8**, faster memory access (decode is memory-bound, so faster too). Cost: **precision loss** (fewer levels → rounding error), but good methods keep it nearly imperceptible.

下面**动手实现最简单的 INT8 量化**(对称量化): 用一个缩放因子把 float 映射到 [-127,127] 的整数, 看**体积减少**和**误差大小**。
Let's **implement the simplest INT8 quantization** (symmetric): map floats to integers in [-127,127] via a scale, and see the **size reduction** and **error**.


In [ ]:
def quantize_int8(x):
    scale = x.abs().max() / 127.0                         # 缩放因子: 让最大绝对值映射到127 / scale to [-127,127]
    q = torch.round(x / scale).clamp(-127, 127).to(torch.int8)   # float → int8(舍入) / round to int8
    return q, scale
def dequantize(q, scale):
    return q.to(torch.float32) * scale                    # int8 → float(近似还原) / approximate restore

W = torch.randn(1000, 1000)                               # 一个权重矩阵 / a weight matrix
q, scale = quantize_int8(W)
W_hat = dequantize(q, scale)                              # 量化再还原 / quantize then dequantize
err = (W - W_hat).abs().mean().item()
print(f"原始 FP32 权重: {W.numel()*4/1e6:.1f} MB")
print(f"量化 INT8 权重: {q.numel()*1/1e6:.1f} MB  ← 体积减到 1/4")
print(f"量化误差(平均绝对): {err:.4f}  (相对权重尺度很小, 几乎无感)")
print(f"原始值示例: {W[0,:4].tolist()}")
print(f"还原值示例: {[round(v,3) for v in W_hat[0,:4].tolist()]}  (略有舍入但很接近)")
# 可视化: 量化前后权重分布几乎重合 / distributions nearly overlap
fig, ax = plt.subplots(figsize=(7,3.5))
ax.hist(W.flatten().numpy(), bins=60, alpha=0.6, label="FP32 原始", color="#39c")
ax.hist(W_hat.flatten().numpy(), bins=60, alpha=0.5, label="INT8 量化还原", color="#e67")
ax.set_title(f"量化前后权重分布几乎重合(误差{err:.4f}) → 4倍压缩, 精度损失极小"); ax.legend()
plt.tight_layout(); plt.show()
print("量化: FP32→INT8 体积1/4(INT4则1/8), 访存更快; 代价是小舍入误差; 配合QLoRA(12.9)可单卡跑大模型")


<a id="4"></a>
## 4. 投机解码、批处理与小结 ⭐ / Speculative Decoding, Batching & Summary

更多系统级优化(面试可展开)：
More system-level optimizations (good to elaborate):
- **投机解码(speculative decoding)**：用一个**小而快的"草稿模型"**一次猜出后面好几个 token, 再让**大模型一次性并行验证**(prefill 是并行的, 验证很快)。猜对的就接受、猜错的回退。因为小模型大多时候猜得对, **几次大模型前向就能产出好几个 token** → 加速 2-3 倍且**输出分布不变**(无损)。
  **Speculative decoding:** a **small fast "draft model"** guesses several next tokens, then the **big model verifies them in parallel** (prefill is parallel, so cheap). Accept correct guesses, roll back wrong ones. Since the draft is usually right, **a few big-model passes yield several tokens** → 2–3× speedup with **identical output distribution** (lossless).
- **连续批处理(continuous batching) + PagedAttention(vLLM)**：服务多个用户请求时, 不同请求长度不一、生成时刻不一。**连续批处理**动态地把新请求拼进正在跑的 batch、完成的及时移出, 让 GPU 始终满载；**PagedAttention** 像操作系统分页一样管理 KV cache 显存, 减少碎片浪费。这是 **vLLM** 等高吞吐推理引擎的核心。
  **Continuous batching + PagedAttention (vLLM):** serving many requests of varying lengths/timings — continuous batching dynamically merges new requests into the running batch and evicts finished ones to keep the GPU saturated; PagedAttention manages KV-cache memory like OS paging to cut fragmentation. Core of high-throughput engines like **vLLM**.
- **FlashAttention**：重写注意力的 GPU 计算, **不把巨大的注意力矩阵写回显存**(分块在快的片上内存里算), 大幅提速且省显存——训练和推理都用。
  **FlashAttention:** rewrite attention's GPU kernel to **avoid materializing the huge attention matrix in memory** (tiled in fast on-chip memory), much faster and memory-lighter — used in training and inference.

```
推理两阶段: prefill(输入并行处理,快,算力密集) + decode(逐token串行,慢,访存密集)
慢的原因: 自回归串行 + 每token过整个巨大模型(访存密集)
KV cache(最重要): 缓存历史token的K/V, 每步只算新token → O(n²)降到O(n); 代价是缓存占显存
量化: FP32→INT8(1/4)/INT4(1/8), 省显存+访存快, 小舍入误差; 配QLoRA单卡跑大模型
投机解码: 小草稿模型猜多个token, 大模型并行验证; 2-3x加速且无损(输出分布不变)
连续批处理+PagedAttention(vLLM): 动态批处理+分页管理KV cache显存 → 高吞吐服务
FlashAttention: 不物化注意力矩阵(片上分块计算), 训练推理都提速省显存
```

### 💡 面试速查 / Interview cheat-sheet
1. **prefill vs decode**: prefill并行快(算力密集), decode串行慢(访存密集)。
   Prefill vs decode: prefill parallel/compute-bound, decode serial/memory-bound.
2. **KV cache**: 缓存历史K/V, 每步只算新token, O(n²)→O(n); 最关键优化。
   KV cache: cache past K/V, compute only the new token; O(n²)→O(n); the key optimization.
3. **量化**: 低位宽(int8/int4)省显存+访存快, 代价小精度损失。
   Quantization: low-bit (int8/int4) saves memory/speeds access; small precision cost.
4. **投机解码**: 小模型猜+大模型并行验证, 2-3x无损加速。
   Speculative decoding: small drafts + big-model parallel verify; 2–3× lossless.
5. **vLLM**: 连续批处理 + PagedAttention(分页管理KV cache) → 高吞吐。
   vLLM: continuous batching + PagedAttention (paged KV cache) → high throughput.

### 🎉 Part 12 完成 / Part 12 Complete
你已**从零**走完现代 NLP 与大模型：RNN/LSTM、注意力、**从零实现的 Transformer**、BERT/GPT/T5、分词(BPE)、预训练与微调、**LoRA**、RLHF/DPO、Prompt 工程、**RAG**、向量数据库、Agent、评估、推理优化。这条线把"大模型黑盒"彻底拆开——从一个 token 怎么被预测, 到一个对话助手怎么被训练、对齐、检索增强、部署上线。这正是当今 AI 岗位最核心的知识体系。
You've gone through modern NLP & LLMs **from scratch**: RNN/LSTM, attention, a **from-scratch Transformer**, BERT/GPT/T5, tokenization (BPE), pretraining & fine-tuning, **LoRA**, RLHF/DPO, prompting, **RAG**, vector DBs, agents, evaluation, inference optimization. This opens the "LLM black box" end to end — from how one token is predicted to how an assistant is trained, aligned, retrieval-augmented, and deployed. The core knowledge base for today's AI roles.
